In [1]:
import sys, os
from pathlib import Path
from pprint import pprint
from copy import deepcopy
from tqdm import tqdm
from collections import defaultdict

%env NX_CUGRAPH_AUTOCONFIG=True
import numpy as np
import torch
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm

from sklearn.metrics import precision_recall_fscore_support#confusion_matrix

import yaml
from tqdm import tqdm

BASE_DIR = Path.cwd().resolve().parent.parent
sys.path.append(str(BASE_DIR))
SCRIPT_DIR = Path.cwd().resolve()
CFG_DIR = SCRIPT_DIR / 'configs'
RESULTS_DIR = SCRIPT_DIR / 'results'


from src.models.lr_ssd.snn__logn_gtv import SNN__LOGN_GTV
from src.models.tucker_decomp.hosvd import hosvd
from src.metrics.estimate_rank import estimate_tucker_rank

from src.metrics.metric_tracker import MetricTracker
from src.multilinear_ops.matricize import matricize
from src.multilinear_ops.tensorize import tensorize
from src.multilinear_ops.mode_product import mode_product

from src.gsp.gsp_vis import draw_graph_signal
from data.server_machine_dataset import SMDMachineChannel
from data.nyc_taxi_dataset import NYCTaxiDataset

plt.style.use('seaborn-v0_8-paper')
# Set the default font family to serif
plt.rcParams['font.family'] = 'serif' 
# Specify a list of font names to try for the serif family
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Palatino'] 

env: NX_CUGRAPH_AUTOCONFIG=True


In [ ]:
def matrix_optimal_hard_sv_threshold(svals: torch.Tensor,
                                    dims: tuple,
                                    return_noise_est: bool = True) -> float:
    """Compute the optimal hard threshold for singular values using Marchenko-Pastur distribution.
    
    Implementation based on:
    Gavish, M., & Donoho, D. L. (2014). The optimal hard threshold for singular
    values is 4/√3.
    Parameters
    ----------
    svals : torch.Tensor
        Singular values of the input data matrix.
    dims : tuple
        Dimensions of the input data matrix (num_rows, num_cols).

    Returns
    -------
    float
        Optimal hard threshold for singular values.
    """
    m, n = dims
    n = min(m, n)
    beta = min(m, n) / max(m, n)

    y_med = torch.median(svals)
    omega_beta = 0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43
    threshold = omega_beta * y_med
    return threshold


def matrix_noise_estimation(svals: torch.Tensor, dims: tuple) -> float:
    """Estimate the noise level in a matrix using Marchenko-Pastur distribution.
    
    Implementation based on:
    Gavish, M., & Donoho, D. L. (2014). The optimal hard threshold for singular
    values is 4/√3.
    Parameters
    ----------
    Y : torch.Tensor
        Input data matrix of shape (num_rows, num_cols) or the singular values

    Returns
    -------
    float
        Estimated noise level.
    """

    m, n = dims
    n = min(m, n)
    beta = min(m, n) / max(m, n)

    S = svals.clone().detach().cpu().sort(descending=True).values
    omega_beta = 0.56 * beta**3 - 0.95 * beta**2 + 1.82 * beta + 1.43
    
    median_singular_value = torch.median(S)
    noise_std = median_singular_value *
    return noise_std.item()


In [9]:
ch = [1]#, 2, 3, 4, 5, 6, 7, 8]

rank_estimates = defaultdict(list)

mchannel = SMDMachineChannel(1,1, center_data=False)
Y = mchannel.Y
Y_tensor = torch.tensor(Y, dtype=torch.float64, device='cuda:1')

for mode in range(Y_tensor.ndim):
    dim = Y_tensor.shape[mode]
    Ym = matricize(Y_tensor, [mode+1])
    svals = torch.linalg.svdvals(Ym)
    noise_level = matrix_noise_estimation(svals, Ym.shape)
    print(f"Mode {mode+1} ({dim} x ..) estimated noise level: {noise_level:.4f}")


Mode 1 (38 x ..) estimated noise level: 0.0905
Mode 2 (20 x ..) estimated noise level: 0.8670
Mode 3 (24 x ..) estimated noise level: 0.7589
Mode 4 (60 x ..) estimated noise level: 0.1465
